In [54]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [55]:
imports_df = pd.read_csv('../outputs/census_imports.csv')
exports_df = pd.read_csv('../outputs/census_exports.csv') 
gdp_df = pd.read_csv('../outputs/gdp_states.csv')

In [56]:
gdp_overall = gdp_df[gdp_df['NAICS'] == '..0'][['STATE/REGION','YEAR','GDP']].rename(columns={'GDP':'OVERALL_GDP'})
gdp_df = gdp_df.merge(gdp_overall, on=['STATE/REGION','YEAR'])
gdp_df['GDP_CAPITA'] = gdp_df['GDP'] / gdp_df['EST_POP']

### Grouping Maps

In [57]:
level_map = dict(gdp_df[['NAICS_LDESC','LEVEL']].drop_duplicates().values)

In [58]:
bea_regions = {
    # New England
    'CT': 'New England', 'ME': 'New England', 'MA': 'New England', 
    'NH': 'New England', 'RI': 'New England', 'VT': 'New England',
    # Mideast
    'DE': 'Mideast', 'DC': 'Mideast', 'MD': 'Mideast', 
    'NJ': 'Mideast', 'NY': 'Mideast', 'PA': 'Mideast',
    # Great Lakes
    'IL': 'Great Lakes', 'IN': 'Great Lakes', 'MI': 'Great Lakes', 
    'OH': 'Great Lakes', 'WI': 'Great Lakes',
    # Plains
    'IA': 'Plains', 'KS': 'Plains', 'MN': 'Plains', 'MO': 'Plains', 
    'NE': 'Plains', 'ND': 'Plains', 'SD': 'Plains',
    # Southeast
    'AL': 'Southeast', 'AR': 'Southeast', 'FL': 'Southeast', 'GA': 'Southeast', 
    'KY': 'Southeast', 'LA': 'Southeast', 'MS': 'Southeast', 'NC': 'Southeast', 
    'SC': 'Southeast', 'TN': 'Southeast', 'VA': 'Southeast', 'WV': 'Southeast',
    # Southwest
    'AZ': 'Southwest', 'NM': 'Southwest', 'OK': 'Southwest', 'TX': 'Southwest',
    # Rocky Mountain
    'CO': 'Rocky Mountain', 'ID': 'Rocky Mountain', 'MT': 'Rocky Mountain', 
    'UT': 'Rocky Mountain', 'WY': 'Rocky Mountain',
    # Far West
    'AK': 'Far West', 'CA': 'Far West', 'HI': 'Far West', 
    'NV': 'Far West', 'OR': 'Far West', 'WA': 'Far West'
}

## **State Growth**

In [59]:
gdp_2010 = gdp_df.loc[gdp_df['YEAR'] == 2010].drop(columns=['YEAR','EST_POP','OVERALL_GDP']).rename(columns={'GDP': 'GDP_2010', 'GDP_CAPITA':'GDP_CAPITA_2010'})
gdp_2024 = gdp_df.loc[gdp_df['YEAR'] == 2024].drop(columns=['YEAR','EST_POP','OVERALL_GDP']).rename(columns={'GDP': 'GDP_2024', 'GDP_CAPITA':'GDP_CAPITA_2024'})

gdp_change = gdp_2010.merge(gdp_2024, on=[
        'REGION_NAME',
        'STATE/REGION',
        'NAICS',
        'NAICS_LDESC',
        'LEVEL'])
gdp_change['GDP_GROWTH'] = (gdp_change['GDP_2024'] - gdp_change['GDP_2010']) / gdp_change['GDP_2010'] * 100
gdp_change['CAPITA_GROWTH'] = (gdp_change['GDP_CAPITA_2024'] - gdp_change['GDP_CAPITA_2010']) / gdp_change['GDP_CAPITA_2010'] * 100

In [60]:
# Top overall GDP Growth
(gdp_change
    .loc[gdp_change['NAICS'] == '..0']
    .sort_values(by='GDP_GROWTH',ascending=False)
    .head(10)[['REGION_NAME','GDP_GROWTH']])

,REGION_NAME,GDP_GROWTH
4048,Utah,153.153564
4324,Washington,135.463418
1104,Idaho,130.035552
828,Florida,129.311231
184,Arizona,127.555752
3128,North Dakota,124.556123
3956,Texas,120.582358
460,Colorado,117.170655
3864,Tennessee,116.966832
2576,Nevada,114.952177


In [61]:
# Top overall GDP Capita Growth
(gdp_change
    .loc[gdp_change['NAICS'] == '..0']
    .sort_values(by='CAPITA_GROWTH',ascending=False)
    .head(10)[['REGION_NAME','CAPITA_GROWTH']])

,REGION_NAME,CAPITA_GROWTH
4048,Utah,100.573538
4324,Washington,100.269975
368,California,97.966539
184,Arizona,92.952054
3128,North Dakota,90.978291
3864,Tennessee,90.164290
2484,Nebraska,87.768602
828,Florida,85.750122
2944,New York,84.956972
460,Colorado,83.047004


## **Industry Growth**

In [63]:
naics_change = (gdp_change
    .groupby('NAICS_LDESC')[['GDP_2010','GDP_CAPITA_2010','GDP_2024','GDP_CAPITA_2024']].sum()
    .reset_index())
naics_change_reg = (gdp_change
    .groupby(['REGION_NAME','STATE/REGION','NAICS_LDESC'])[['GDP_2010','GDP_CAPITA_2010','GDP_2024','GDP_CAPITA_2024']].sum()
    .reset_index())

naics_change['GDP_GROWTH'] = (naics_change['GDP_2024'] - naics_change['GDP_2010']) / naics_change['GDP_2010'] * 100
naics_change['CAPITA_GROWTH'] = (naics_change['GDP_CAPITA_2024'] - naics_change['GDP_CAPITA_2010']) / naics_change['GDP_CAPITA_2010'] * 100
naics_change_reg['GDP_GROWTH'] = (naics_change_reg['GDP_2024'] - naics_change_reg['GDP_2010']) / naics_change_reg['GDP_2010'] * 100
naics_change_reg['CAPITA_GROWTH'] = (naics_change_reg['GDP_CAPITA_2024'] - naics_change_reg['GDP_CAPITA_2010']) / naics_change_reg['GDP_CAPITA_2010'] * 100

naics_change['LEVEL'] = naics_change['NAICS_LDESC'].map(level_map)
naics_change_reg['LEVEL'] = naics_change_reg['NAICS_LDESC'].map(level_map)
naics_change_reg['BEA_REGION'] = naics_change_reg['STATE/REGION'].map(bea_regions)

#### Sector Group Growth (GDP)

In [64]:
(naics_change
    .loc[(naics_change['LEVEL'] == 0) & (naics_change['NAICS_LDESC'] != 'Nondurable goods manufacturing') & (naics_change['NAICS_LDESC'] != 'Durable goods manufacturing')]
    .sort_values(by='GDP_GROWTH',ascending=False)
    .head(10)[['NAICS_LDESC','GDP_GROWTH']])

,NAICS_LDESC,GDP_GROWTH
11,"Arts, entertainment, recreation, accommodation...",131.401985
68,Professional and business services,115.505229
26,"Finance, insurance, real estate, rental, and l...",110.054785
81,Trade,103.027208
83,Transportation and utilities,102.301365
20,"Educational services, health care, and social ...",94.019843
41,Manufacturing and information,75.894220
50,Natural resources and mining,49.067661


#### Sector & Industry Group Growth (GDP per capita)

In [66]:
# Sector Growth
(naics_change
    .loc[naics_change['LEVEL'] == 2]
    .sort_values(by='CAPITA_GROWTH',ascending=False)
    .head(5)[['NAICS_LDESC','CAPITA_GROWTH']])

,NAICS_LDESC,CAPITA_GROWTH
16,Construction,114.542618
1,Accommodation and food services,113.447333
25,Finance and insurance,97.840384
10,"Arts, entertainment, and recreation",94.103452
84,Transportation and warehousing,93.818659


In [67]:
# Subsector Growth
(naics_change
    .loc[naics_change['LEVEL'] == 3]
    .sort_values(by='CAPITA_GROWTH',ascending=False)
    .head(10)[['NAICS_LDESC','CAPITA_GROWTH']])

,NAICS_LDESC,CAPITA_GROWTH
30,"Funds, trusts, and other financial vehicles",525.523047
17,"Data processing, hosting, and other informatio...",199.677688
87,Warehousing and storage,147.761705
91,Wood product manufacturing,130.330088
28,Food services and drinking places,115.949372
61,Pipeline transportation,115.879306
56,Other transportation and support activities,113.742607
47,"Monetary Authorities- central bank, credit int...",111.161367
0,Accommodation,108.754291
77,Social assistance,106.321844


#### Sector Group GDP (per capita) Growth, by Region/State

In [69]:
# Top State Sector Growth (Construction)
(naics_change_reg
    .loc[(naics_change_reg['LEVEL'] == 2) & (naics_change_reg['NAICS_LDESC'] == 'Construction')]
    .sort_values(by='CAPITA_GROWTH',ascending=False)
    .head(10)[['REGION_NAME','NAICS_LDESC','CAPITA_GROWTH']])

,REGION_NAME,NAICS_LDESC,CAPITA_GROWTH
4064,Utah,Construction,260.115516
200,Arizona,Construction,217.458417
476,Colorado,Construction,195.961871
3420,Oregon,Construction,180.726290
3052,North Carolina,Construction,179.642223
936,Georgia,Construction,177.951012
1120,Idaho,Construction,173.087164
1304,Indiana,Construction,165.085298
2132,Minnesota,Construction,156.903669
3236,Ohio,Construction,155.720604


In [70]:
# Top State Sector Growth (Accomodation and food services)
(naics_change_reg
    .loc[(naics_change_reg['LEVEL'] == 2) & (naics_change_reg['NAICS_LDESC'] == 'Accommodation and food services')]
    .sort_values(by='CAPITA_GROWTH',ascending=False)
    .head(10)[['REGION_NAME','NAICS_LDESC','CAPITA_GROWTH']])

,REGION_NAME,NAICS_LDESC,CAPITA_GROWTH
3865,Tennessee,Accommodation and food services,180.942438
1749,Maine,Accommodation and food services,180.004651
3773,South Dakota,Accommodation and food services,179.635287
2669,New Hampshire,Accommodation and food services,172.280575
2393,Montana,Accommodation and food services,169.240855
1105,Idaho,Accommodation and food services,157.370286
4141,Vermont,Accommodation and food services,155.069208
369,California,Accommodation and food services,143.559702
3037,North Carolina,Accommodation and food services,140.487142
3681,South Carolina,Accommodation and food services,138.631754


In [71]:
# Top State Sector Growth (Finance and insurance)
(naics_change_reg
    .loc[(naics_change_reg['LEVEL'] == 2) & (naics_change_reg['NAICS_LDESC'] == 'Finance and insurance')]
    .sort_values(by='CAPITA_GROWTH',ascending=False)
    .head(10)[['REGION_NAME','NAICS_LDESC','CAPITA_GROWTH']])

,REGION_NAME,NAICS_LDESC,CAPITA_GROWTH
2509,Nebraska,Finance and insurance,234.349816
209,Arizona,Finance and insurance,166.318297
4073,Utah,Finance and insurance,165.337647
3245,Ohio,Finance and insurance,164.342400
1405,Iowa,Finance and insurance,154.542239
1129,Idaho,Finance and insurance,140.545375
3061,North Carolina,Finance and insurance,137.068137
2969,New York,Finance and insurance,135.322494
3981,Texas,Finance and insurance,134.636163
301,Arkansas,Finance and insurance,132.988696


In [72]:
# Top State Sector Growth (Arts, entertainment, and recreation)
(naics_change_reg
    .loc[(naics_change_reg['LEVEL'] == 2) & (naics_change_reg['NAICS_LDESC'] == 'Arts, entertainment, and recreation')]
    .sort_values(by='CAPITA_GROWTH',ascending=False)
    .head(10)[['REGION_NAME','NAICS_LDESC','CAPITA_GROWTH']])

,REGION_NAME,NAICS_LDESC,CAPITA_GROWTH
1850,Maryland,"Arts, entertainment, and recreation",208.059291
286,Arkansas,"Arts, entertainment, and recreation",169.178201
3874,Tennessee,"Arts, entertainment, and recreation",168.389867
3414,Oregon,"Arts, entertainment, and recreation",167.654795
470,Colorado,"Arts, entertainment, and recreation",145.497140
4058,Utah,"Arts, entertainment, and recreation",144.454043
3966,Texas,"Arts, entertainment, and recreation",138.576399
194,Arizona,"Arts, entertainment, and recreation",137.349464
1206,Illinois,"Arts, entertainment, and recreation",135.671661
3138,North Dakota,"Arts, entertainment, and recreation",135.190959


In [73]:
# Top State Sector Growth (Transportation and warehousing)
(naics_change_reg
    .loc[(naics_change_reg['LEVEL'] == 2) & (naics_change_reg['NAICS_LDESC'] == 'Transportation and warehousing')]
    .sort_values(by='CAPITA_GROWTH',ascending=False)
    .head(10)[['REGION_NAME','NAICS_LDESC','CAPITA_GROWTH']])

,REGION_NAME,NAICS_LDESC,CAPITA_GROWTH
3396,Oklahoma,Transportation and warehousing,196.636923
452,California,Transportation and warehousing,168.572987
728,Delaware,Transportation and warehousing,149.765093
544,Colorado,Transportation and warehousing,139.015791
2108,Michigan,Transportation and warehousing,126.606415
2016,Massachusetts,Transportation and warehousing,124.171180
912,Florida,Transportation and warehousing,122.846590
268,Arizona,Transportation and warehousing,118.704528
4408,Washington,Transportation and warehousing,114.870653
636,Connecticut,Transportation and warehousing,114.660403


### Growth Contribution

In [75]:
gdp_df.sort_values(by=['STATE/REGION','NAICS','YEAR'])

gdp_df['YoY CHANGE'] = gdp_df.groupby(['STATE/REGION','NAICS'])['GDP'].pct_change() * 100
gdp_df['GDP_SHARE'] = gdp_df['GDP'] / gdp_df['OVERALL_GDP']
gdp_df['GDP_SHARE_PY'] = gdp_df.groupby(['STATE/REGION','NAICS'])['GDP_SHARE'].shift(1)
gdp_df['GROWTH_CONTRIBUTION'] = gdp_df['GDP_SHARE_PY'] * gdp_df['YoY CHANGE']

gdp_df['BEA_REGION'] = gdp_df['STATE/REGION'].map(bea_regions)

#### Sector Growth Contribution

In [ ]:
# Top Sector Growth Contribution, Overall
(gdp_df
    .loc[gdp_df['LEVEL'] == 2]
    .groupby(['NAICS_LDESC'])['GROWTH_CONTRIBUTION']
    .sum()
    .reset_index()
    .sort_values(by='GROWTH_CONTRIBUTION',ascending=False)
   .head(10))

,NAICS_LDESC,GROWTH_CONTRIBUTION
15,Real estate and rental and leasing,484.301519
6,Finance and insurance,320.827994
7,Government and government enterprises,320.754713
8,Health care and social assistance,300.331210
14,"Professional, scientific, and technical services",287.436798
11,Manufacturing,270.049234
16,Retail trade,259.975347
19,Wholesale trade,222.452614
4,Construction,200.931086
0,Accommodation and food services,156.034546


In [78]:
# Top Sector Growth Contribution, by BEA Region
(gdp_df
    .loc[gdp_df['LEVEL'] == 2]
    .groupby(['BEA_REGION','NAICS_LDESC'])['GROWTH_CONTRIBUTION']
    .sum()
    .reset_index()
    .sort_values(by='GROWTH_CONTRIBUTION',ascending=False)
    .head(10))

,BEA_REGION,NAICS_LDESC,GROWTH_CONTRIBUTION
135,Southeast,Real estate and rental and leasing,120.454410
131,Southeast,Manufacturing,77.802670
127,Southeast,Government and government enterprises,74.694645
128,Southeast,Health care and social assistance,70.960121
136,Southeast,Retail trade,68.268587
126,Southeast,Finance and insurance,64.358611
115,Rocky Mountain,Real estate and rental and leasing,62.566024
86,Plains,Finance and insurance,62.416786
134,Southeast,"Professional, scientific, and technical services",61.379487
15,Far West,Real estate and rental and leasing,60.900153


In [80]:
# Top Sector Growth Contribution, by State/Region
(gdp_df
    .loc[(gdp_df['LEVEL'] == 2) & (gdp_df['BEA_REGION'] == 'Southeast')]
    .groupby(['REGION_NAME','NAICS_LDESC'])['GROWTH_CONTRIBUTION']
    .sum()
    .reset_index()
    .sort_values(by='GROWTH_CONTRIBUTION',ascending=False))

,REGION_NAME,NAICS_LDESC,GROWTH_CONTRIBUTION
55,Florida,Real estate and rental and leasing,20.141081
175,South Carolina,Real estate and rental and leasing,14.088187
75,Georgia,Real estate and rental and leasing,12.859590
195,Tennessee,Real estate and rental and leasing,10.555090
155,North Carolina,Real estate and rental and leasing,10.490633
...,...,...,...
12,Alabama,"Mining, quarrying, and oil and gas extraction",-0.393834
132,Mississippi,"Mining, quarrying, and oil and gas extraction",-0.638464
92,Kentucky,"Mining, quarrying, and oil and gas extraction",-1.784988
32,Arkansas,"Mining, quarrying, and oil and gas extraction",-2.016576


#### Subsector Growth Contribution

In [82]:
# Top Subsector Growth Contribution, Overall
(gdp_df
    .loc[gdp_df['LEVEL'] == 3]
    .groupby(['NAICS_LDESC'])['GROWTH_CONTRIBUTION']
    .sum()
    .reset_index()
    .sort_values(by='GROWTH_CONTRIBUTION',ascending=False)
    .head(10))

,NAICS_LDESC,GROWTH_CONTRIBUTION
38,Real estate,400.880353
23,"Monetary Authorities- central bank, credit int...",141.617244
3,Ambulatory health care services,130.515449
19,Insurance carriers and related activities,109.375972
1,Administrative and support services,102.392033
14,Food services and drinking places,101.210951
18,Hospitals,89.598404
39,Rental and leasing services and lessors of non...,52.539888
0,Accommodation,51.953982
12,Farms,48.492237


In [83]:
# Top Subsector Growth Contribution, by BEA Region
(gdp_df
    .loc[gdp_df['LEVEL'] == 3]
    .groupby(['BEA_REGION','NAICS_LDESC'])['GROWTH_CONTRIBUTION']
    .sum()
    .reset_index()
    .sort_values(by='GROWTH_CONTRIBUTION',ascending=False)
    .head(10))

,BEA_REGION,NAICS_LDESC,GROWTH_CONTRIBUTION
338,Southeast,Real estate,100.934606
288,Rocky Mountain,Real estate,53.082129
38,Far West,Real estate,51.160154
188,New England,Real estate,44.594500
238,Plains,Real estate,42.017279
138,Mideast,Real estate,36.477767
388,Southwest,Real estate,35.563309
303,Southeast,Ambulatory health care services,33.590879
88,Great Lakes,Real estate,31.142792
219,Plains,Insurance carriers and related activities,29.545956


In [84]:
# Top Subsector Growth Contribution, by State/Region
(gdp_df
    .loc[(gdp_df['LEVEL'] == 3) & (gdp_df['BEA_REGION'] == 'Southeast')]
    .groupby(['REGION_NAME','NAICS_LDESC'])['GROWTH_CONTRIBUTION']
    .sum()
    .reset_index()
    .sort_values(by='GROWTH_CONTRIBUTION',ascending=False))

,REGION_NAME,NAICS_LDESC,GROWTH_CONTRIBUTION
138,Florida,Real estate,17.266280
438,South Carolina,Real estate,12.167353
188,Georgia,Real estate,9.395739
388,North Carolina,Real estate,9.072629
488,Tennessee,Real estate,8.865533
...,...,...,...
221,Kentucky,Mining (except oil and gas),-1.386660
77,Arkansas,Oil and gas extraction,-1.601231
358,North Carolina,Computer and electronic product manufacturing,-1.772675
281,Louisiana,Petroleum and coal products manufacturing,-1.774200


## **International Trade**

In [85]:
manufacturing_codes = {'31','32','33'}

### Export vs Import Overall Value

In [86]:
(exports_df
    .loc[(~exports_df['EXP_NAICS'].isin(manufacturing_codes)) & (exports_df['EXP_LEVEL'] == 2)]
    .groupby('EXP_NAICS_DESC')['EXPORT_VAL'].sum()
    .reset_index()
    .sort_values(by='EXPORT_VAL',ascending=False)
)

,EXP_NAICS_DESC,EXPORT_VAL
2,MANUFACTURED GOODS,22033255898083
3,"OIL, GAS, MINERALS AND ORES",1651606926188
0,AGRICULTURE AND LIVESTOCK PRODUCTS,1311885347179
7,WASTE AND SCRAP,390616740902
6,USED OR SECOND-HAND MERCHANDISE,308203447976
4,OTHER SPECIAL CLASSIFICATION PROVISIONS,271627062640
5,PUBLISHERS' COMMODITIES,1819334668
1,GOODS RETURNED TO CANADA (EXPORTS ONLY); U.S. ...,1781362428


In [87]:
(imports_df
    .loc[(~imports_df['IMP_NAICS'].isin(manufacturing_codes)) & (imports_df['IMP_LEVEL'] == 2)]
    .groupby('IMP_NAICS_DESC')['IMPORT_VAL'].sum()
    .reset_index()
    .sort_values(by='IMPORT_VAL',ascending=False)
)

,IMP_NAICS_DESC,IMPORT_VAL
2,MANUFACTURED GOODS,34813491007197
3,"OIL, GAS, MINERALS AND ORES",3243802405942
1,GOODS RETURNED TO CANADA (EXPORTS ONLY); U.S. ...,1208799322253
0,AGRICULTURE AND LIVESTOCK PRODUCTS,1027013969460
6,USED OR SECOND-HAND MERCHANDISE,188096039984
7,WASTE AND SCRAP,111216907396
4,OTHER SPECIAL CLASSIFICATION PROVISIONS,77227526193
5,PUBLISHERS' COMMODITIES,76801255


### Export vs Import Overall Growth

In [89]:
export_change = (exports_df
    .loc[(exports_df['YEAR'] == 2010) & (exports_df['EXP_LEVEL'] == 2) & (~exports_df['EXP_NAICS'].isin(manufacturing_codes))]
    .drop(columns=['GDP_CODE'])
    .groupby(['STATE/REGION','YEAR'])['EXPORT_VAL'].sum()
    .reset_index()
    .rename(columns={'EXPORT_VAL':'EXPORTS_2010'})
    .drop(columns=['YEAR'])
).merge((exports_df
    .loc[(exports_df['YEAR'] == 2024) & (exports_df['EXP_LEVEL'] == 2) & (~exports_df['EXP_NAICS'].isin(manufacturing_codes))]
    .drop(columns=['GDP_CODE'])
    .groupby(['STATE/REGION','YEAR'])['EXPORT_VAL'].sum()
    .reset_index()
    .rename(columns={'EXPORT_VAL':'EXPORTS_2024'})
    .drop(columns=['YEAR'])
),on=['STATE/REGION'])

export_change = (export_change
    .merge(gdp_change.loc[gdp_change['NAICS'] == '..0'][['STATE/REGION','REGION_NAME','GDP_2010','GDP_2024']], on=['STATE/REGION']))

export_change['EXPORT_2010_EXP'] = export_change['EXPORTS_2010'] / export_change['GDP_2010'] * 100
export_change['EXPORT_2024_EXP'] = export_change['EXPORTS_2024'] / export_change['GDP_2024'] * 100

import_change = (imports_df
    .loc[(imports_df['YEAR'] == 2010) & (imports_df['IMP_LEVEL'] == 2) & (~imports_df['IMP_NAICS'].isin(manufacturing_codes))]
    .drop(columns=['GDP_CODE'])
    .groupby(['STATE/REGION','YEAR'])['IMPORT_VAL'].sum()
    .reset_index()
    .rename(columns={'IMPORT_VAL':'IMPORTS_2010'})
    .drop(columns=['YEAR'])
).merge((imports_df
    .loc[(imports_df['YEAR'] == 2024) & (imports_df['IMP_LEVEL'] == 2) & (~imports_df['IMP_NAICS'].isin(manufacturing_codes))]
    .drop(columns=['GDP_CODE'])
    .groupby(['STATE/REGION','YEAR'])['IMPORT_VAL'].sum()
    .reset_index()
    .rename(columns={'IMPORT_VAL':'IMPORTS_2024'})
    .drop(columns=['YEAR'])
),on=['STATE/REGION'])

import_change = (import_change
    .merge(gdp_change.loc[gdp_change['NAICS'] == '..0'][['STATE/REGION','REGION_NAME','GDP_2010','GDP_2024']], on=['STATE/REGION']))

import_change['IMPORT_2010_EXP'] = import_change['IMPORTS_2010'] / import_change['GDP_2010'] * 100
import_change['IMPORT_2024_EXP'] = import_change['IMPORTS_2024'] / import_change['GDP_2024'] * 100

export_change['EXP_GROWTH_PP'] = export_change['EXPORT_2024_EXP'] - export_change['EXPORT_2010_EXP']
import_change['IMP_GROWTH_PP'] = import_change['IMPORT_2024_EXP'] - import_change['IMPORT_2010_EXP']

In [90]:
(export_change
    .loc[export_change['EXP_GROWTH_PP'] > 0][['REGION_NAME','EXPORT_2010_EXP','EXPORT_2024_EXP','EXP_GROWTH_PP']]
    .sort_values(by='EXP_GROWTH_PP',ascending=False))

,REGION_NAME,EXPORT_2010_EXP,EXPORT_2024_EXP,EXP_GROWTH_PP
17,Louisiana,18.260723,26.229884,7.969161
31,New Mexico,1.817821,8.155764,6.337943
16,Kentucky,11.608451,16.244750,4.636299
14,Indiana,10.103862,11.593636,1.489774
27,North Dakota,7.102604,8.533834,1.431230
49,Wyoming,2.617434,4.034165,1.416731
0,Alaska,7.743648,8.261404,0.517756
24,Mississippi,8.676119,8.760493,0.084374
19,Maryland,3.238766,3.290097,0.051331


In [92]:
(import_change
    .loc[import_change['IMP_GROWTH_PP'] > 0][['REGION_NAME','IMPORT_2010_EXP','IMPORT_2024_EXP','IMP_GROWTH_PP']]
    .sort_values(by='IMP_GROWTH_PP',ascending=False))

,REGION_NAME,IMPORT_2010_EXP,IMPORT_2024_EXP,IMP_GROWTH_PP
16,Kentucky,19.256255,31.936781,12.680526
14,Indiana,11.644243,20.524847,8.880603
1,Alabama,8.393496,11.888144,3.494647
41,Tennessee,18.635073,21.457450,2.822377
35,Oklahoma,4.203702,6.793071,2.589369
13,Illinois,16.407235,18.983541,2.576306
31,New Mexico,3.728764,6.075135,2.346371
0,Alaska,2.751807,5.072836,2.321029
21,Michigan,22.392320,24.629328,2.237008
32,Nevada,4.832002,6.995337,2.163334


### Export vs Import Sector Growth

In [94]:
export_change_sec = (exports_df
    .loc[(exports_df['YEAR'] == 2010) & (exports_df['EXP_LEVEL'] == 2) & (~exports_df['EXP_NAICS'].isin(manufacturing_codes))]
    .drop(columns=['GDP_CODE','EXP_LEVEL'])
    .groupby(['EXP_NAICS_DESC','YEAR'])['EXPORT_VAL'].sum()
    .reset_index()
    .rename(columns={'EXPORT_VAL':'EXPORTS_2010'})
    .drop(columns=['YEAR'])
).merge((exports_df
    .loc[(exports_df['YEAR'] == 2024) & (exports_df['EXP_LEVEL'] == 2) & (~exports_df['EXP_NAICS'].isin(manufacturing_codes))]
    .drop(columns=['GDP_CODE','EXP_LEVEL'])
    .groupby(['EXP_NAICS_DESC','YEAR'])['EXPORT_VAL'].sum()
    .reset_index()
    .rename(columns={'EXPORT_VAL':'EXPORTS_2024'})
    .drop(columns=['YEAR'])
),on=['EXP_NAICS_DESC'])

import_change_sec = (imports_df
    .loc[(imports_df['YEAR'] == 2010) & (imports_df['IMP_LEVEL'] == 2) & (~imports_df['IMP_NAICS'].isin(manufacturing_codes))]
    .drop(columns=['GDP_CODE','IMP_LEVEL'])
    .groupby(['IMP_NAICS_DESC','YEAR'])['IMPORT_VAL'].sum()
    .reset_index()
    .rename(columns={'IMPORT_VAL':'IMPORTS_2010'})
    .drop(columns=['YEAR'])
).merge((imports_df
    .loc[(imports_df['YEAR'] == 2024) & (imports_df['IMP_LEVEL'] == 2) & (~imports_df['IMP_NAICS'].isin(manufacturing_codes))]
    .drop(columns=['GDP_CODE','IMP_LEVEL'])
    .groupby(['IMP_NAICS_DESC','YEAR'])['IMPORT_VAL'].sum()
    .reset_index()
    .rename(columns={'IMPORT_VAL':'IMPORTS_2024'})
    .drop(columns=['YEAR'])
),on=['IMP_NAICS_DESC'])

export_change_sec['EXP_SECT_GROWTH'] = (export_change_sec['EXPORTS_2024'] - export_change_sec['EXPORTS_2010']) / export_change_sec['EXPORTS_2010'] * 100
import_change_sec['IMP_SECT_GROWTH'] = (import_change_sec['IMPORTS_2024'] - import_change_sec['IMPORTS_2010']) / import_change_sec['IMPORTS_2010'] * 100

In [95]:
export_change_sec.sort_values(by='EXP_SECT_GROWTH',ascending=False)

,EXP_NAICS_DESC,EXPORTS_2010,EXPORTS_2024,EXP_SECT_GROWTH
3,"OIL, GAS, MINERALS AND ORES",26839724508,206611379158,669.796944
5,USED OR SECOND-HAND MERCHANDISE,8248885121,23553993880,185.541543
4,OTHER SPECIAL CLASSIFICATION PROVISIONS,14360607763,22213950828,54.686704
2,MANUFACTURED GOODS,1085010034555,1621039383892,49.403170
0,AGRICULTURE AND LIVESTOCK PRODUCTS,67678871967,88092796224,30.162920
6,WASTE AND SCRAP,29529063019,27780619670,-5.921093
1,GOODS RETURNED TO CANADA (EXPORTS ONLY); U.S. ...,158339345,96625976,-38.975385


In [96]:
import_change_sec.sort_values(by='IMP_SECT_GROWTH',ascending=False)

,IMP_NAICS_DESC,IMPORTS_2010,IMPORTS_2024,IMP_SECT_GROWTH
5,USED OR SECOND-HAND MERCHANDISE,6401861883,16212879308,153.252563
1,GOODS RETURNED TO CANADA (EXPORTS ONLY); U.S. ...,40911036473,101152819285,147.250688
0,AGRICULTURE AND LIVESTOCK PRODUCTS,42675771044,82142379992,92.480131
2,MANUFACTURED GOODS,1513019754298,2844330313487,87.990296
6,WASTE AND SCRAP,5254454964,7484104328,42.433504
4,OTHER SPECIAL CLASSIFICATION PROVISIONS,5736257778,5564617910,-2.992192
3,"OIL, GAS, MINERALS AND ORES",285913190402,182454552323,-36.185332


## **Tennessee**

In [98]:
gdp_change['NATIONAL_RANK'] = gdp_change.groupby('NAICS_LDESC')['CAPITA_GROWTH'].rank(method="min",ascending=False)

In [111]:
# Sectors where Tennessee ranked top 5 in GDP per capita growth
(gdp_change
    .loc[(gdp_change['REGION_NAME'] == 'Tennessee') & (gdp_change['LEVEL'] == 2) & (gdp_change['NATIONAL_RANK'] <= 5)])

,REGION_NAME,STATE/REGION,NAICS,NAICS_LDESC,LEVEL,GDP_2010,GDP_CAPITA_2010,GDP_2024,GDP_CAPITA_2024,GDP_GROWTH,CAPITA_GROWTH,NATIONAL_RANK
3873,Tennessee,TN,22,Utilities,2,2.056700e+09,323.608556,6.353500e+09,876.188806,208.917197,170.755760,3.0
3898,Tennessee,TN,44-45,Retail trade,2,1.860430e+10,2927.267297,4.380030e+10,6040.345092,135.431056,106.347575,4.0
3939,Tennessee,TN,71,"Arts, entertainment, and recreation",2,4.295700e+09,675.900847,1.315420e+10,1814.049388,206.217846,168.389867,3.0
3942,Tennessee,TN,72,Accommodation and food services,2,7.757200e+09,1220.545674,2.486490e+10,3429.030775,220.539628,180.942438,1.0


In [115]:
lvl_3_secgroups = ['Durable goods manufacturing','Nondurable goods manufacturing']
(gdp_df
    .loc[(gdp_df['REGION_NAME'] == 'Tennessee') & (gdp_df['LEVEL'] == 0) & (~gdp_df['NAICS_LDESC'].isin(lvl_3_secgroups))])

,REGION_NAME,STATE/REGION,NAICS,NAICS_LDESC,LEVEL,YEAR,GDP,EST_POP,OVERALL_GDP,GDP_CAPITA,YoY CHANGE,GDP_SHARE,GDP_SHARE_PY,GROWTH_CONTRIBUTION,BEA_REGION
3913,Tennessee,TN,"52,53","Finance, insurance, real estate, rental, and l...",0,2010,4.290940e+10,6355518,2.586576e+11,6751.518916,NaN,0.165893,NaN,NaN,Southeast
3922,Tennessee,TN,"54,55,56",Professional and business services,0,2010,2.674690e+10,6355518,2.586576e+11,4208.453190,NaN,0.103407,NaN,NaN,Southeast
3931,Tennessee,TN,"61,62","Educational services, health care, and social ...",0,2010,3.017330e+10,6355518,2.586576e+11,4747.575257,NaN,0.116653,NaN,NaN,Southeast
3938,Tennessee,TN,"71,72","Arts, entertainment, recreation, accommodation...",0,2010,1.205300e+10,6355518,2.586576e+11,1896.462255,NaN,0.046598,NaN,NaN,Southeast
3950,Tennessee,TN,"11,21",Natural resources and mining,0,2010,2.214000e+09,6355518,2.586576e+11,348.358702,NaN,0.008560,NaN,NaN,Southeast
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74318,Tennessee,TN,"71,72","Arts, entertainment, recreation, accommodation...",0,2025,3.960830e+10,7315076,5.898175e+11,5414.612234,4.180278,0.067153,0.067746,0.283196,Southeast
74330,Tennessee,TN,"11,21",Natural resources and mining,0,2025,3.908600e+09,7315076,5.898175e+11,534.321175,-1.322898,0.006627,0.007058,-0.009337,Southeast
74331,Tennessee,TN,"42,44-45",Trade,0,2025,8.690240e+10,7315076,5.898175e+11,11879.903914,7.343103,0.147338,0.144258,1.059299,Southeast
74332,Tennessee,TN,"22,48-49",Transportation and utilities,0,2025,3.438730e+10,7315076,5.898175e+11,4700.880756,3.238503,0.058302,0.059352,0.192213,Southeast
